# Model Evaluation
Evaluating saved model on a held-out dataset using metrics appropriate to your target type. Keep evaluation data separate from training data.

In [1]:
#import libraries 
import seaborn as sns

from sklearn.metrics import (
 accuracy_score,
 classification_report,
 confusion_matrix,
 mean_absolute_error,
 mean_squared_error,
 r2_score
)
from sklearn.model_selection import train_test_split



In [2]:
from pathlib import Path
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)

MODEL_PATH = PROJECT_ROOT / 'models' / 'irrigation_model.joblib'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

TARGET_COLUMN = 'Irrigation_Need'
print("Paths initialized successfully!")
print("Project Root:", PROJECT_ROOT)
print("Model Path:", MODEL_PATH)
print("Processed Data Directory:", PROCESSED_DATA_DIR)

Paths initialized successfully!
Project Root: G:\AIOT\smart-irrigation
Model Path: G:\AIOT\smart-irrigation\models\irrigation_model.joblib
Processed Data Directory: G:\AIOT\smart-irrigation\data\processed


In [3]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

TARGET_COLUMN = 'Irrigation_Need'
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_PATH = PROJECT_ROOT / "models" / "irrigation_model.joblib"

# Prefer an independent test file; otherwise reconstruct the training notebook's held-out split.
test_file = PROCESSED_DATA_DIR / 'test.csv'
uses_dedicated_test_file = test_file.is_file()
if not uses_dedicated_test_file:
  test_file = PROCESSED_DATA_DIR / 'irrigation_prediction_processed.csv'
  if not test_file.is_file():
    raise FileNotFoundError(
        f'Processed dataset not found: {test_file}. Run notebook 02 first.'
    )

if not MODEL_PATH.is_file():
  raise FileNotFoundError(f'Model not found at {MODEL_PATH}.')

data = pd.read_csv(test_file)

if TARGET_COLUMN not in data.columns:
  raise ValueError(
      f"Target column '{TARGET_COLUMN}' not found in dataset columns:"
      f' {list(data.columns)}'
  )

X = data.drop(columns=[TARGET_COLUMN]).select_dtypes(include='number')
y = data[TARGET_COLUMN]
if uses_dedicated_test_file:
  X_test, y_test = X, y
else:
  _, X_test, _, y_test = train_test_split(
      X, y, test_size=0.2, random_state=42
  )

# Load model
model = joblib.load(MODEL_PATH)

# Validate feature consistency if the model exposes feature names
if hasattr(model, 'feature_names_in_'):
  missing_cols = set(model.feature_names_in_) - set(X_test.columns)
  if missing_cols:
    raise ValueError(f'Missing expected features in test data: {missing_cols}')
  X_test = X_test[model.feature_names_in_]

predictions = model.predict(X_test)
print(f'Evaluated {len(X_test):,} held-out rows from {test_file.name}.')

# Evaluate based on target data type
if y_test.dtype.kind in 'if':
  metrics = {
      'mae': mean_absolute_error(y_test, predictions),
      'rmse': mean_squared_error(y_test, predictions) ** 0.5,
      'r2': r2_score(y_test, predictions),
  }
  print('Regression Metrics:')
  for metric, value in metrics.items():
    print(f'  {metric.upper():<5}: {value:.4f}')
else:
  print(f'Accuracy: {accuracy_score(y_test, predictions):.3f}\n')
  print('Classification Report:')
  print(classification_report(y_test, predictions))

Evaluated 2,000 held-out rows from irrigation_prediction_processed.csv.
Accuracy: 0.728

Classification Report:
              precision    recall  f1-score   support

        High       0.68      0.29      0.40        59
         Low       0.77      0.83      0.80      1204
      Medium       0.64      0.59      0.61       737

    accuracy                           0.73      2000
   macro avg       0.70      0.57      0.61      2000
weighted avg       0.72      0.73      0.72      2000

